Importing Libraries
We're bringing in tools (pandas, numpy, scikit-learn) to help us load the data, clean it, and get it ready for training.

Loading the Data
We load a CSV file called gpin_data.csv, which contains many packet flows and their behavior — like size, timing, and if it was part of an attack.

Encoding the Target
The target column tells us whether a packet is normal or an attack (like "BENIGN", "SYN_FLOOD", etc.).
Since machine learning doesn’t understand text, we convert these names into numbers using a tool called LabelEncoder.

Separating Features from Labels

Features = the columns that describe the packet’s behavior (like packet size, duration, etc.).

Target = the type of packet (attack or normal).
We separate these two because the model will learn to predict the target using the features.

Scaling the Features
Different features might be in different ranges (some small, some large), so we scale everything to be in the same range using StandardScaler. This helps the model learn better and faster.

Making Sequences for LSTM
LSTM models learn from sequences (like watching 10 packets in a row instead of just 1).
So, we split the data into chunks of 10 packets (SEQ_LEN = 10), making each chunk one input sample.

Trimming the Data
We only keep full sequences — no incomplete chunks.

Reshaping the Data
We organize the data into a shape that the LSTM model expects:
(number of sequences, 10 steps per sequence, number of features per step)

Target for Each Sequence
Since each sample is 10 packets long, we assign the label of the 10th packet as the label for the whole sequence.
(Example: if the last packet in that group is an attack, we treat the whole sequence as an attack.)

Splitting into Training and Testing Sets
We divide the data:

80% is used for training the model.

20% is saved for testing how well the model performs.
We make sure the test set has a balanced mix of all types of attacks using stratify.

Final Outputs

We print the shapes of the training data.

And also show how each attack type was converted into a number using the label encoder.

🎯 Why are we doing this?
Because our goal is to build a smart model that can look at packet behavior and decide if it’s normal or an attack. But before we can train it, we need to carefully prepare the data so the model has the best chance of learning correctly. This entire cell is just the data preparation phase of the GPIN system.

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Load CSV
df = pd.read_csv("gpin_data.csv")

# Encode the target
label_encoder = LabelEncoder()
df['target_encoded'] = label_encoder.fit_transform(df['target'])

# Extract features and target
feature_cols = df.drop(columns=['target', 'target_encoded']).columns.tolist()
X = df[feature_cols].values
y = df['target_encoded'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define SEQ_LEN
SEQ_LEN = 10
num_sequences = len(X_scaled) // SEQ_LEN

# Trim data
X_seq = X_scaled[:num_sequences * SEQ_LEN]
y_seq = y[:num_sequences * SEQ_LEN]

# Reshape into (N, 10, features)
X_lstm = X_seq.reshape(num_sequences, SEQ_LEN, len(feature_cols))

# Target = label of last flow in each sequence
y_lstm = y_seq.reshape(num_sequences, SEQ_LEN)[:, -1]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_lstm, y_lstm, test_size=0.2, random_state=42, stratify=y_lstm
)

print("✅ Sequence-based preprocessing done.")
print("🔷 X_train shape:", X_train.shape)
print("🔶 y_train shape:", y_train.shape)
print("🔖 Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

✅ Sequence-based preprocessing done.
🔷 X_train shape: (40000, 10, 15)
🔶 y_train shape: (40000,)
🔖 Label mapping: {'BENIGN': 0, 'IP_SPOOF': 1, 'SYN_FLOOD': 2, 'TCP_FLOOD': 3, 'UDP_FLOOD': 4}


In this code cell, we are building and training an LSTM-based deep learning model to detect whether a sequence of network packets is normal or part of a cyberattack. First, we extract important shape details like how many packets are in a sequence (`timesteps`), how many features describe each packet (`num_features`), and how many types of packet classes we want to predict (`num_classes`). Then, using Keras' Functional API, we define the model structure: the input layer takes in sequences of packet features, which are passed through an LSTM layer that helps the model understand patterns across time. The LSTM output is sent to a dense layer with 32 neurons and ReLU activation to further process the learned features, and finally to an output layer with softmax activation that gives a probability for each attack type or benign class.

After building the model, we compile it using the Adam optimizer (which helps the model learn efficiently), sparse categorical crossentropy as the loss function (suitable for multi-class classification with integer labels), and accuracy as a metric to evaluate how well the model is performing. To prevent overfitting, we add an early stopping mechanism that stops training if the validation loss doesn’t improve for 3 consecutive epochs and restores the best version of the model. We then train the model using `model.fit`, providing the training data and labels, using 20% of it for validation, and running the training for up to 12 epochs with batches of 64 samples at a time. Finally, we print the model summary to review its architecture and total parameters. This entire setup is designed to train a robust sequence-based classifier that can learn the behavior of packet flows and accurately predict whether a given flow is malicious or benign.
⤵

In [4]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Extract shape info
timesteps = X_train.shape[1]        # 10
num_features = X_train.shape[2]     # e.g., 15–20
num_classes = len(np.unique(y_train))

# Build LSTM model using Functional API
input_layer = Input(shape=(timesteps, num_features))
lstm_out = LSTM(64, return_sequences=False)(input_layer)
dense_out = Dense(32, activation='relu')(lstm_out)   # Latent representation
output_layer = Dense(num_classes, activation='softmax')(dense_out)

model = Model(inputs=input_layer, outputs=output_layer)

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Add early stopping to avoid overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=12,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# Summary
model.summary()

Epoch 1/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8132 - loss: 0.4693 - val_accuracy: 0.8758 - val_loss: 0.2172
Epoch 2/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8778 - loss: 0.2201 - val_accuracy: 0.8761 - val_loss: 0.2180
Epoch 3/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8798 - loss: 0.2163 - val_accuracy: 0.8750 - val_loss: 0.2157
Epoch 4/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8804 - loss: 0.2164 - val_accuracy: 0.8759 - val_loss: 0.2137
Epoch 5/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8832 - loss: 0.2102 - val_accuracy: 0.8759 - val_loss: 0.2137
Epoch 6/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8815 - loss: 0.2123 - val_accuracy: 0.8759 - val_loss: 0.2155
Epoch 7/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8786 - loss: 0.2176 - val_accuracy: 0.8748 - val_loss: 0.2149
Epoch 8/12
500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8790 - loss: 0.2152 - val_accuracy: 0.

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 15)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,177 (266.32 KB)

 Trainable params: 22,725 (88.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 45,452 (177.55 KB)

In this part of our project, we are using two smart systems to work together like a team — one is a neural network (LSTM) that learns how packets behave over time, and the other is a decision-making system (XGBoost) that makes the final call about whether a packet is safe or dangerous.

Here’s what we’re doing step by step:

Extracting knowledge from the LSTM model:
We already trained an LSTM model earlier that can understand patterns in network traffic. But instead of using its final answer (output), we want to take the “smart thoughts” it has just before making a decision — these are found in a hidden layer with 32 numbers (we call these the feature vector or dense features).

Creating features from data:
Now we pass our training and test data through that tool to get the LSTM’s learned features.
So now, instead of raw packet numbers, we have smart compressed summaries of each packet, ready to be classified.

Training the XGBoost classifier:
Next, we use another model called XGBoost. It’s like a decision tree army that’s very good at figuring out rules. We give it the LSTM features from training data and teach it to recognize what kind of attack (or no attack) the data belongs to.

Making predictions:
Once XGBoost is trained, we give it new packet data (from the test set), and it tries to guess whether the packets are safe or types of attacks.

Checking how well it did:
Finally, we print out:
The accuracy (how many predictions were correct overall),
And a classification report, which shows how well it predicted each attack type.

💡 Why are we doing this?
Because LSTM is good at understanding sequences and XGBoost is good at making decisions. So we let LSTM understand the behavior, and let XGBoost judge what kind of packet it is — just like a detective explains what happened, and a judge gives the final verdict.

In [5]:
from tensorflow.keras.models import Model
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# 🔹 Build the feature extractor model (extract Dense(32) layer)
feature_extractor = Model(inputs=model.input, outputs=model.layers[-2].output)

# 🔹 Get LSTM-learned features for both train and test sets
lstm_features_train = feature_extractor.predict(X_train)
lstm_features_test = feature_extractor.predict(X_test)

print(f"✅ LSTM feature vectors extracted: {lstm_features_train.shape}")

# 🔹 Train the XGBoost model
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='mlogloss'
)
xgb.fit(lstm_features_train, y_train)

# 🔹 Predict using XGBoost
y_pred = xgb.predict(lstm_features_test)

# 🔹 Evaluate performance
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📄 Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
✅ LSTM feature vectors extracted: (40000, 32)
✅ Accuracy: 0.8803

📄 Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      2000
    IP_SPOOF       0.63      1.00      0.77      2000
   SYN_FLOOD       1.00      0.40      0.57      2000
   TCP_FLOOD       1.00      1.00      1.00      2000
   UDP_FLOOD       1.00      1.00      1.00      2000

    accuracy                           0.88     10000
   macro avg       0.93      0.88      0.87     10000
weighted avg       0.93      0.88      0.87     10000



In [6]:
# Decode numeric labels to readable strings
true_labels = label_encoder.inverse_transform(y_test)
pred_labels = label_encoder.inverse_transform(y_pred)

print("🔍 Sample Predictions:\n")
for i in range(50):  # change to 50 or 100 if needed
    print(f"{i+1:02d}) 🟢 True: {true_labels[i]:<15} | 🔵 Predicted: {pred_labels[i]}")


🔍 Sample Predictions:

01) 🟢 True: UDP_FLOOD       | 🔵 Predicted: UDP_FLOOD
02) 🟢 True: UDP_FLOOD       | 🔵 Predicted: UDP_FLOOD
03) 🟢 True: SYN_FLOOD       | 🔵 Predicted: SYN_FLOOD
04) 🟢 True: TCP_FLOOD       | 🔵 Predicted: TCP_FLOOD
05) 🟢 True: BENIGN          | 🔵 Predicted: BENIGN
06) 🟢 True: IP_SPOOF        | 🔵 Predicted: IP_SPOOF
07) 🟢 True: SYN_FLOOD       | 🔵 Predicted: IP_SPOOF
08) 🟢 True: IP_SPOOF        | 🔵 Predicted: IP_SPOOF
09) 🟢 True: BENIGN          | 🔵 Predicted: BENIGN
10) 🟢 True: IP_SPOOF        | 🔵 Predicted: IP_SPOOF
11) 🟢 True: SYN_FLOOD       | 🔵 Predicted: SYN_FLOOD
12) 🟢 True: SYN_FLOOD       | 🔵 Predicted: IP_SPOOF
13) 🟢 True: UDP_FLOOD       | 🔵 Predicted: UDP_FLOOD
14) 🟢 True: BENIGN          | 🔵 Predicted: BENIGN
15) 🟢 True: SYN_FLOOD       | 🔵 Predicted: IP_SPOOF
16) 🟢 True: IP_SPOOF        | 🔵 Predicted: IP_SPOOF
17) 🟢 True: IP_SPOOF        | 🔵 Predicted: IP_SPOOF
18) 🟢 True: BENIGN          | 🔵 Predicted: BENIGN
19) 🟢 True: TCP_FLOOD       | 🔵 Predicted: 

In [25]:
#setting up ollama (local llm)
import requests
import json

def send_to_gemma(prompt):
    try:
        url = "http://127.0.0.1:11434/api/chat"
        headers = {"Content-Type": "application/json"}
        data = {
            "model": "gemma3",
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        }
        response = requests.post(url, headers=headers, data=json.dumps(data))
        res = response.json()

        # Log response content
        gemma_response = res.get("message", {}).get("content", "")
        print(f"\n🧠 GEMMA’s Recommendation:\n{gemma_response}\n")

        # Log to file
        with open("gpin_attack_logs.txt", "a", encoding="utf-8") as f:
            f.write("\n\n========== NEW ATTACK DETECTED ==========\n")
            f.write(f"{prompt}\n\n")
            f.write("🧠 GEMMA Suggestion:\n")
            f.write(gemma_response + "\n")

    except Exception as e:
        print(f"⚠️ Error querying Gemma: {e}")

In [28]:
import random

# 1. Pick a random test sample
random_idx = random.randint(0, len(X_test) - 1)

# 2. Get the predicted label
pred_numeric = y_pred[random_idx]
pred_label = label_encoder.inverse_transform([pred_numeric])[0]

# 3. Only continue if it's an actual attack
if pred_label != "BENIGN":
        # Build prompt from sample features
        flow_features = X_test[random_idx][-1]  # since shape is (1, seq_len, features)
        feature_summary = "\n".join([
            f"- {col}: {round(val, 4)}"
            for col, val in zip(feature_cols, flow_features)
        ])
        prompt = f"""
🚨 [ATTACK DETECTED]: {pred_label}

The model has detected a packet flow that appears malicious.

Behavioral indicators:
{feature_summary}

Why is this attack happening and suggest 5 ways to prevent or mitigate it from a cybersecurity and system-level perspective in brief.
"""
        send_to_gemma(prompt)

else:
    print(f"✅ Safe packet detected: {pred_label}. No prompt generated.")



🧠 GEMMA’s Recommendation:
Okay, let's break down this TCP Flood attack detection and outline prevention/mitigation strategies.

**Understanding the Attack**

The model’s indicators strongly suggest a TCP Flood attack. Here’s what they mean:

* **`flow_duration: -0.193`**:  A significantly short flow duration is a classic sign – attackers quickly establish and then terminate connections to overwhelm the target.
* **`flow_byts_s: -0.4098`, `flow_pkts_s: -0.4462`**:  Low data transfer rates within these flows are also consistent with flooding; attackers aren't actually trying to *use* the connection.
* **`syn_flag_cnt: -0.8894`**: A very low number of SYN packets indicates a lack of legitimate connection requests. Floods often start with a massive SYN bombardment.
* **Overall:** The pattern points to a volume of packets being sent to the target, designed to consume resources and make the service unavailable.

**Why is this happening?**

TCP Flood attacks are commonly used by attackers to

In [8]:
##part two

In [ ]:
import os
from datetime import datetime

log_file = "gpin_attack_logs.txt"

# Ensure fresh file each run (optional)
if os.path.exists(log_file):
    os.remove(log_file)

for i in range(50):
    # 1. Prepare one input sample
    lstm_input = X_test[i].reshape(1, X_test.shape[1], X_test.shape[2])
    lstm_features = feature_extractor.predict(lstm_input)

    # 2. Predict using XGBoost
    pred_class = xgb.predict(lstm_features)[0]
    pred_label = label_encoder.inverse_transform([pred_class])[0]

    if pred_label != "BENIGN":
        # Build prompt from sample features
        flow_features = sample[0][0]  # since shape is (1, seq_len, features)
        feature_summary = "\n".join([
            f"- {col}: {round(val, 4)}"
            for col, val in zip(feature_cols, flow_features)
        ])
        prompt = f"""
🚨 [ATTACK DETECTED]: {pred_label}

The model has detected a packet flow that appears malicious.

Behavioral indicators:
{feature_summary}

Explain the nature of this attack and suggest ways to prevent or mitigate it from a cybersecurity and system-level perspective.
"""
        send_to_gemma(prompt)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step

==================== [🚨 ATTACK DETECTED] ====================
🕒 Timestamp: 2025-07-07 17:25:19
🔖 Type: UDP_FLOOD

📊 Flow Characteristics:
- flow_duration: 0.0
- flow_byts_s: 11430000000000.0
- flow_pkts_s: 10000000000.0
- tot_fwd_pkts: 1.0
- tot_bwd_pkts: 0.0
- fwd_pkt_len_mean: 1143.0
- bwd_pkt_len_mean: 0.0
- pkt_len_var: 0.0
- flow_iat_mean: 0.0
- fwd_iat_max: 0.0
- bwd_iat_max: 0.0
- syn_flag_cnt: 0.0
- ack_flag_cnt: 0.0
- pkt_size_avg: 1143.0
- init_fwd_win_byts: 0.0

💡 Prompt:
Explain the most likely cause of this UDP_FLOOD attack and suggest specific, technical mitigation strategies for a system admin (rate-limiting, firewall rules, application-level hardening, etc.)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

==================== [🚨 ATTACK DETECTED] ====================
🕒 Timestamp: 2025-07-07 17:25:19
🔖 Type: UDP_FLOOD

📊 Flow Characteristics:
- flow_duration: 463.2633
- flow_byts_s: 2.2687
- flow_pkts_s: 0.0043
- tot_fwd_pkts: 2.0
- tot_bwd_